# MIDI composer: fine-tune a small LLM to write classical piano music from a brief

**Standalone version.** Upload this file to Colab (File > Upload notebook) and run it; nothing else is needed. The library code is in the "Library code" section below. It is generated from `src/midicomposer/` in the repo by `scripts/build_standalone.py`, so make code changes there.

**Before running**
1. Runtime > Change runtime type > **GPU** (T4 works; A100/L4 is much faster).
2. Put your MIDI files in Google Drive at `MyDrive/MIDI/` (or upload a zip when the MIDI cell asks).
3. Optional but needed for real training: add a Colab Secret (key icon in the left sidebar) named `GEMINI_API_KEY`. Without it, descriptions fall back to metadata-only one-liners, which are fine for a smoke test only.

Leave `SMOKE_TEST = True` for the first run (12 pieces, 30 training steps, short generation) to check that every stage works, then set it to `False`.

Everything slow (encoded text, descriptions, checkpoints, outputs) is cached in `MyDrive/midi-composer/`, so after a disconnect you can rerun from the top and it picks up where it left off. If Drive won't mount ("credential propagation was unsuccessful"), rerun the first cell and tick every permission in the Google popup, using the same account as Colab. Otherwise the notebook continues with local storage, and the last cell downloads the outputs as a zip.

In [ ]:
import glob, json, os, sys

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = False
REPO_DIR = os.getcwd()

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        USE_DRIVE = True
    except Exception as e:
        print(f'Google Drive did not mount: {e}\n'
              'Continuing with storage local to this runtime; it is lost when the runtime disconnects.\n'
              'To use Drive: rerun this cell, pick the same Google account as Colab, and tick every permission.')
    !pip install -q pretty_midi mido google-genai datasets
    !apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null

os.makedirs('midicomposer_lib/midicomposer', exist_ok=True)
sys.path.insert(0, os.path.abspath('midicomposer_lib'))

## Library code

The `midicomposer` package, written to `midicomposer_lib/` so the cells below can import it. Run these cells; don't edit them here (they are regenerated from the repo's `src/midicomposer/`).

In [ ]:
%%writefile midicomposer_lib/midicomposer/__init__.py



In [ ]:
%%writefile midicomposer_lib/midicomposer/codec.py
"""MIDI <-> text codec for score-aligned solo piano MIDI.

Format:

    <|piece_start|>
    <|meta|> key=F_Minor ts=12/8 bars=262 spb=12
    <|plan|> Exposition:1-65 Development:66-135 ...      (optional; ignored by decode)
    bar 1 @0.00 q126 ts=12/8
    R v6 0:C5:6 v5 6:Ab4.C5:6
    L v4 0:F2.C3:36
    ...
    <|piece_end|>

Bar lines: bar number, `@` fractional position in the piece, `q` average quarter-note
BPM over the bar (bar start times survive decoding; rubato within a bar does not),
`ts=` only when the time signature changes, and `len=` only when the bar's length in
grid steps differs from what its time signature implies (cadenzas, irregular bars).

Hand lines (`R`, `L`): note events `onset:pitches:duration`, onset relative to the bar,
both in grid steps (`spb` steps per quarter note). Simultaneous notes sharing onset,
duration, and velocity are joined with `.`. A `vN` token sets the velocity bin for the
events that follow it; every hand line that has notes starts with one.
"""

import bisect
import re
from collections import defaultdict

import mido
import numpy as np
import pretty_midi

SPB = 12
VEL_BINS = 8
TICKS_PER_STEP = 40
PIANO_RANGE = (21, 108)

_MIDO_KEYS = [
    "C", "Db", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B",
    "Cm", "C#m", "Dm", "Ebm", "Em", "Fm", "F#m", "Gm", "G#m", "Am", "Bbm", "Bm",
]

HANDS = ("R", "L")
TRACK_NAMES = {"R": "Piano right", "L": "Piano left"}


def _hand(inst):
    return "L" if "left" in inst.name.lower() else "R"


_KS_MAJOR = [6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88]
_KS_MINOR = [6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17]


def _key_name(pm):
    """Main key from the first key signature. These files never mark minor mode, so choose
    between the signature's major key and its relative minor: the final bar's bass note
    decides when it is one of the two tonics, otherwise Krumhansl-Schmuckler correlation."""
    if not pm.key_signature_changes:
        return "unknown"
    marked = pm.key_signature_changes[0].key_number
    if marked >= 12:
        return pretty_midi.key_number_to_key_name(marked).replace(" ", "_")
    major = marked
    minor = (major + 9) % 12
    notes = [n for inst in pm.instruments if not inst.is_drum for n in inst.notes]
    last_downbeat = pm.get_downbeats()[-1] if len(pm.get_downbeats()) else 0
    final = [n for n in notes if n.start >= last_downbeat] or notes[-8:]
    bass = min(final, key=lambda n: n.pitch).pitch % 12
    if bass in (major, minor) and major != minor:
        is_minor = bass == minor
    else:
        hist = np.zeros(12)
        for n in notes:
            hist[n.pitch % 12] += n.end - n.start
        corr = lambda profile, tonic: np.corrcoef(hist, np.roll(profile, tonic))[0, 1]
        is_minor = corr(_KS_MINOR, minor) > corr(_KS_MAJOR, major)
    number = minor + 12 if is_minor else major
    return pretty_midi.key_number_to_key_name(number).replace(" ", "_")


def _ts_steps(ts, spb):
    num, den = map(int, ts.split("/"))
    return round(num * 4 / den * spb)


def encode(pm, spb=SPB, vel_bins=VEL_BINS):
    tpb = pm.resolution
    step = tpb / spb
    notes = [(inst, n) for inst in pm.instruments if not inst.is_drum for n in inst.notes]
    end_tick = max(pm.time_to_tick(n.end) for _, n in notes)

    bar_ticks = sorted({pm.time_to_tick(t) for t in pm.get_downbeats()})
    bar_ticks = [t for t in bar_ticks if t < end_tick] or [0]
    n_bars = len(bar_ticks)
    bar_ticks.append(max(end_tick, bar_ticks[-1] + 1))
    bar_steps = [round(t / step) for t in bar_ticks]

    ts_changes = sorted(
        (pm.time_to_tick(ts.time), f"{ts.numerator}/{ts.denominator}") for ts in pm.time_signature_changes
    ) or [(0, "4/4")]
    ts_ticks = [t for t, _ in ts_changes]

    events = {h: [defaultdict(set) for _ in range(n_bars)] for h in HANDS}
    for inst, n in notes:
        s = round(pm.time_to_tick(n.start) / step)
        dur = max(1, round(pm.time_to_tick(n.end) / step) - s)
        b = min(max(bisect.bisect_right(bar_steps, s) - 1, 0), n_bars - 1)
        vel = min(vel_bins - 1, n.velocity * vel_bins // 128)
        events[_hand(inst)][b][(s - bar_steps[b], dur, vel)].add(n.pitch)

    lines = ["<|piece_start|>", f"<|meta|> key={_key_name(pm)} ts={ts_changes[0][1]} bars={n_bars} spb={spb}"]
    prev_ts = None
    for b in range(n_bars):
        t0, t1 = bar_ticks[b], bar_ticks[b + 1]
        ts = ts_changes[max(bisect.bisect_right(ts_ticks, t0) - 1, 0)][1]
        seconds = pm.tick_to_time(t1) - pm.tick_to_time(t0)
        tempo = round(60 * (t1 - t0) / tpb / seconds) if seconds > 0 else 120
        fields = [f"bar {b + 1}", f"@{b / n_bars:.2f}", f"q{tempo}"]
        if ts != prev_ts:
            fields.append(f"ts={ts}")
        prev_ts = ts
        length = bar_steps[b + 1] - bar_steps[b]
        if length != _ts_steps(ts, spb) and b < n_bars - 1:
            fields.append(f"len={length}")
        lines.append(" ".join(fields))
        for h in HANDS:
            toks, cur_vel = [h], None
            for (on, dur, vel), pitches in sorted(events[h][b].items()):
                if vel != cur_vel:
                    toks.append(f"v{vel}")
                    cur_vel = vel
                names = ".".join(pretty_midi.note_number_to_name(p) for p in sorted(pitches))
                toks.append(f"{on}:{names}:{dur}")
            lines.append(" ".join(toks))
    lines.append("<|piece_end|>")
    return "\n".join(lines)


_FIELD = re.compile(r"(\w+)=(\S+)")
_PITCH = r"[A-G][#b]?-?\d"
_NOTE = re.compile(rf"^(\d+):({_PITCH}(?:\.{_PITCH})*):(\d+)$")


def decode(text, vel_bins=VEL_BINS):
    """Parse codec text into a mido.MidiFile. Malformed lines and tokens are skipped."""
    spb, ts, tempo, key = SPB, "4/4", 120, None
    bar_start, bar_len, in_bar = 0, None, False
    tempo_events, ts_events = [], []
    notes = {h: [] for h in HANDS}

    for line in text.splitlines():
        tag, _, rest = line.strip().partition(" ")
        fields = dict(_FIELD.findall(rest))
        if tag == "<|meta|>":
            spb = int(fields.get("spb", spb))
            ts = fields.get("ts", ts)
            key = fields.get("key")
        elif tag == "bar":
            if in_bar:
                bar_start += bar_len
            in_bar = True
            if "ts" in fields and fields["ts"] != (ts_events[-1][1] if ts_events else None):
                ts = fields["ts"]
                ts_events.append((bar_start, ts))
            elif not ts_events:
                ts_events.append((bar_start, ts))
            try:
                bar_len = int(fields["len"]) if "len" in fields else _ts_steps(ts, spb)
            except ValueError:
                bar_len = _ts_steps("4/4", spb)
            q = re.search(r"\bq(\d+(?:\.\d+)?)\b", rest)
            if q and float(q[1]) > 0:
                tempo = float(q[1])
            tempo_events.append((bar_start, tempo))
        elif tag in HANDS and in_bar:
            vel = vel_bins // 2
            for tok in rest.split():
                if re.fullmatch(r"v\d+", tok):
                    vel = min(int(tok[1:]), vel_bins - 1)
                    continue
                m = _NOTE.match(tok)
                if not m:
                    continue
                on, names, dur = int(m[1]), m[2], int(m[3])
                velocity = min(127, vel * (128 // vel_bins) + 128 // vel_bins // 2)
                for name in names.split("."):
                    try:
                        pitch = pretty_midi.note_name_to_number(name)
                    except Exception:
                        continue
                    if 0 <= pitch <= 127:
                        notes[tag].append((bar_start + on, max(1, dur), pitch, velocity))

    tpb = spb * TICKS_PER_STEP
    mid = mido.MidiFile(ticks_per_beat=tpb)
    meta = [(s * TICKS_PER_STEP, 0, mido.MetaMessage("set_tempo", tempo=mido.bpm2tempo(t))) for s, t in tempo_events]
    for s, sig in ts_events:
        num, den = map(int, sig.split("/"))
        meta.append((s * TICKS_PER_STEP, 0, mido.MetaMessage("time_signature", numerator=num, denominator=den)))
    try:
        key_number = pretty_midi.key_name_to_key_number(key.replace("_", " "))
        meta.append((0, 0, mido.MetaMessage("key_signature", key=_MIDO_KEYS[key_number])))
    except Exception:
        pass
    mid.tracks.append(_to_track(meta))
    for h in HANDS:
        evs = []
        for on, dur, pitch, vel in notes[h]:
            evs.append((on * TICKS_PER_STEP, 1, mido.Message("note_on", note=pitch, velocity=vel)))
            evs.append(((on + dur) * TICKS_PER_STEP, 0, mido.Message("note_off", note=pitch, velocity=0)))
        track = _to_track(evs)
        track.insert(0, mido.MetaMessage("track_name", name=TRACK_NAMES[h]))
        mid.tracks.append(track)
    return mid


def _to_track(events):
    track, now = mido.MidiTrack(), 0
    for tick, _, msg in sorted(events, key=lambda e: (e[0], e[1])):
        track.append(msg.copy(time=tick - now))
        now = tick
    track.append(mido.MetaMessage("end_of_track", time=0))
    return track


def transpose_key(key, semis):
    try:
        number = pretty_midi.key_name_to_key_number(key.replace("_", " "))
    except Exception:
        return key
    mode = number // 12
    return pretty_midi.key_number_to_key_name((number % 12 + semis) % 12 + 12 * mode).replace(" ", "_")


def transpose(text, semis, piano_range=PIANO_RANGE):
    """Shift every pitch and the key by `semis`. Returns None if any note leaves the piano range."""
    lo, hi = piano_range
    lines = []
    for line in text.splitlines():
        tag = line.split(" ", 1)[0]
        if tag in HANDS:
            toks = []
            for tok in line.split(" "):
                m = _NOTE.match(tok)
                if m:
                    pitches = [pretty_midi.note_name_to_number(p) + semis for p in m[2].split(".")]
                    if not all(lo <= p <= hi for p in pitches):
                        return None
                    tok = f"{m[1]}:{'.'.join(pretty_midi.note_number_to_name(p) for p in pitches)}:{m[3]}"
                toks.append(tok)
            line = " ".join(toks)
        elif tag == "<|meta|>":
            line = re.sub(r"key=(\S+)", lambda m: f"key={transpose_key(m[1], semis)}", line)
        lines.append(line)
    return "\n".join(lines)


In [ ]:
%%writefile midicomposer_lib/midicomposer/features.py
"""Musical features that ground the description LLM."""

import mido
import numpy as np
import pretty_midi

from .codec import _key_name


def embedded_metadata(path):
    """Title, composer, tempo marking, etc. from track-0 text events (piano-midi.de convention)."""
    mid = mido.MidiFile(path)
    out = []
    for msg in mid.tracks[0]:
        if msg.type == "track_name":
            out.append(msg.name)
        elif msg.type == "text":
            out.append(msg.text)
    skip = ("fertiggestellt", "erstellt", "veröffentlicht", "update", "normierung", "dauer", "copyright", "©")
    return [s.strip() for s in out if s.strip() and not any(w in s.lower() for w in skip)][:6]


def extract_features(path):
    pm = pretty_midi.PrettyMIDI(path)
    notes = [n for inst in pm.instruments if not inst.is_drum for n in inst.notes]
    downbeats = list(pm.get_downbeats()) + [pm.get_end_time()]
    _, tempi = pm.get_tempo_changes()
    per_bar = []
    for b in range(len(downbeats) - 1):
        bn = [n for n in notes if downbeats[b] <= n.start < downbeats[b + 1]]
        per_bar.append({
            "bar": b + 1,
            "notes": len(bn),
            "vel": round(float(np.mean([n.velocity for n in bn]))) if bn else 0,
            "lo": min(n.pitch for n in bn) if bn else None,
            "hi": max(n.pitch for n in bn) if bn else None,
        })
    key = _key_name(pm).replace("_", " ")
    return {
        "metadata": embedded_metadata(path),
        "key": key,
        "time_signatures": sorted({f"{t.numerator}/{t.denominator}" for t in pm.time_signature_changes}),
        "tempo_bpm_range": [round(float(np.percentile(tempi, 5))), round(float(np.percentile(tempi, 95)))],
        "duration_s": round(pm.get_end_time()),
        "n_bars": len(per_bar),
        "per_bar": per_bar,
    }


In [ ]:
%%writefile midicomposer_lib/midicomposer/describe.py
"""Composition-brief descriptions of each piece from an audio-capable LLM (Gemini)."""

import json
import os
import shutil
import subprocess
import time

from .features import extract_features

DEFAULT_SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
COMPOSERS = {
    "albeniz": "Albéniz", "bach": "Bach", "balakire": "Balakirev", "beethoven": "Beethoven",
    "borodin": "Borodin", "brahms": "Brahms", "burgm": "Burgmüller", "chopin": "Chopin",
    "clementi": "Clementi", "debussy": "Debussy", "godowsky": "Godowsky", "granados": "Granados",
    "grieg": "Grieg", "haydn": "Haydn", "liszt": "Liszt", "mendelssohn": "Mendelssohn",
    "moszkowski": "Moszkowski", "mozart": "Mozart", "mussorgsk": "Mussorgsky", "rachmanin": "Rachmaninoff",
    "ravel": "Ravel", "schubert": "Schubert", "schumann": "Schumann", "sinding": "Sinding",
    "tschaikowsky": "Tchaikovsky", "tchaikovsky": "Tchaikovsky",
}
TEMPO_WORDS = (
    "adagio", "allegr", "andant", "grave", "larghetto", "largo", "lento", "maestoso", "moderato",
    "presto", "vivace", "vivo", "tempo", "agitato", "con moto", "langsam", "lebhaft", "rasch", "mässig",
)
VARIANTS = ("brief", "paragraph", "detailed", "styled_brief", "styled_paragraph", "styled_detailed")

PROMPT = """You are writing a commission brief that asks a composer to write this piece of
solo piano music. You are given a recording, the score's embedded metadata, and per-bar
analysis (note count, mean MIDI velocity, lowest/highest MIDI pitch).

Return JSON with exactly these keys:
- "brief": one imperative sentence ("Write a ...").
- "paragraph": 3-5 sentences on character, mood, tempo, meter, form, texture, and difficulty.
- "detailed": a section-by-section brief of the emotional and technical progression, giving
  approximate positions in the piece ("around 40% of the way through ..."): harmonic language,
  melodic character, figuration, texture, dynamics, climaxes, and the ending.
- "styled_brief", "styled_paragraph", "styled_detailed": the same three, but also naming the
  composer and the style or genre to emulate ("in the manner of Chopin's nocturnes").
- "sections": a list of {"name": str, "bars": [first, last]} covering bars 1..n_bars in order,
  using musical names (Introduction, Exposition, Theme, Variation 1, Trio, Coda, ...).

Rules: never name the specific piece, its title, or its catalog number. Do not name the key
or the exact tempo in the prose (they are supplied separately); relative terms like "the
relative major" or "a slower middle section" are fine. Write as instructions, not as analysis."""


def render_audio(midi_path, out_dir, soundfont=DEFAULT_SOUNDFONT, sample_rate=22050):
    """Render with FluidSynth, then compress to mono MP3 if ffmpeg is available."""
    os.makedirs(out_dir, exist_ok=True)
    stem = os.path.join(out_dir, os.path.splitext(os.path.basename(midi_path))[0])
    mp3, wav = stem + ".mp3", stem + ".wav"
    if os.path.exists(mp3):
        return mp3
    subprocess.run(["fluidsynth", "-ni", "-F", wav, "-r", str(sample_rate), soundfont, midi_path],
                   check=True, capture_output=True)
    if shutil.which("ffmpeg"):
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav, "-ac", "1", "-b:a", "64k", mp3], check=True)
        os.remove(wav)
        return mp3
    return wav


def describe(midi_path, client, model, audio_dir):
    features = extract_features(midi_path)
    audio = client.files.upload(file=render_audio(midi_path, audio_dir))
    while audio.state.name == "PROCESSING":
        time.sleep(2)
        audio = client.files.get(name=audio.name)
    response = client.models.generate_content(
        model=model,
        contents=[PROMPT, json.dumps(features), audio],
        config={"response_mime_type": "application/json"},
    )
    client.files.delete(name=audio.name)
    desc = json.loads(response.text)
    missing = [k for k in VARIANTS + ("sections",) if k not in desc]
    if missing:
        raise ValueError(f"response missing {missing}")
    return {"file": os.path.basename(midi_path), **desc}


def fallback_description(midi_path):
    """Metadata-only description for smoke-testing the pipeline without an API key."""
    f = extract_features(midi_path)
    meta = f["metadata"]
    composer = next((name for m in meta for key, name in COMPOSERS.items() if key in m.lower()), None)
    marking = next((m for m in meta if any(w in m.lower() for w in TEMPO_WORDS) and len(m) < 40), "moderato")
    brief = f"Write a solo piano piece of about {f['n_bars']} bars, marked {marking}."
    styled = brief[:-1] + (f", in the style of {composer}." if composer else ".")
    return {
        "file": os.path.basename(midi_path),
        "brief": brief, "paragraph": brief, "detailed": brief,
        "styled_brief": styled, "styled_paragraph": styled, "styled_detailed": styled,
        "sections": [],
        "fallback": True,
    }


def describe_all(midi_paths, out_jsonl, client=None, model=None, audio_dir=None, retries=3):
    """Append one JSON line per piece; already-described files are skipped, so this resumes."""
    done = set()
    if os.path.exists(out_jsonl):
        with open(out_jsonl) as fh:
            done = {json.loads(line)["file"] for line in fh if line.strip()}
    for i, path in enumerate(midi_paths):
        name = os.path.basename(path)
        if name in done:
            continue
        for attempt in range(retries):
            try:
                rec = describe(path, client, model, audio_dir) if client else fallback_description(path)
                break
            except Exception as e:
                print(f"{name}: attempt {attempt + 1} failed: {e}")
                time.sleep(5 * (attempt + 1))
        else:
            continue
        with open(out_jsonl, "a") as fh:
            fh.write(json.dumps(rec) + "\n")
        print(f"[{i + 1}/{len(midi_paths)}] {name}")


In [ ]:
%%writefile midicomposer_lib/midicomposer/dataset.py
"""Turn encoded pieces + descriptions into prompt/completion training windows.

Each piece is cut into windows. The first window's completion is the header (piece start,
meta, plan) plus the opening bars; later windows give the header and the most recent bars
as context and ask for the next bars. The same `make_prompt` is used at generation time.
"""

import random
import re

from .codec import transpose
from .describe import VARIANTS

SYSTEM = "You are a composer of classical solo piano music. You write scores in a compact text notation."


def approx_tokens(text):
    return len(text) / 1.25


def split_piece(text):
    """Return (meta_line, bars) where each bar is its 'bar' line plus its hand lines."""
    meta, bars, cur = None, [], []
    for line in text.splitlines():
        if line.startswith("<|meta|>"):
            meta = line
        elif line.startswith("bar "):
            if cur:
                bars.append("\n".join(cur))
            cur = [line]
        elif line[:2] in ("R ", "L ") or line in ("R", "L"):
            if cur:
                cur.append(line)
    if cur:
        bars.append("\n".join(cur))
    return meta, bars


def plan_line(sections):
    if not sections:
        return None
    parts = []
    for s in sections:
        try:
            a, b = s["bars"]
            parts.append(f"{re.sub(r'[^A-Za-z0-9]+', '_', s['name']).strip('_')}:{int(a)}-{int(b)}")
        except (KeyError, TypeError, ValueError):
            continue
    return "<|plan|> " + " ".join(parts) if parts else None


def meta_fields(meta):
    return dict(re.findall(r"(\w+)=(\S+)", meta or ""))


def make_prompt(description, key=None, meter=None, header=None, context_bars=None, next_bar=None):
    """Chat messages for either the opening window (header=None) or a continuation."""
    lines = ["Compose a classical solo piano piece."]
    if key:
        lines.append(f"Key: {key.replace('_', ' ')}")
    if meter:
        lines.append(f"Meter: {meter}")
    lines += ["", description.strip()]
    if header is not None:
        lines += ["", f"Continue the score from bar {next_bar}. Header and most recent bars:", *header]
        lines += context_bars or []
    else:
        lines += ["", "Write the score header and the opening bars."]
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\n".join(lines)}]


def windows(text, sections=None, measure=approx_tokens, target_budget=3000, context_budget=1200,
            max_target_bars=32, max_context_bars=8):
    """Yield (header, context_bars, target_text, next_bar_number) tuples covering the whole piece."""
    meta, bars = split_piece(text)
    plan = plan_line(sections)
    header = [meta] + ([plan] if plan else [])
    start = 0
    while start < len(bars):
        end, used = start, 0
        while end < len(bars) and end - start < max_target_bars:
            cost = measure(bars[end])
            if end > start and used + cost > target_budget:
                break
            used += cost
            end += 1
        ctx, used = [], 0
        for b in reversed(bars[max(0, start - max_context_bars):start]):
            cost = measure(b)
            if used + cost > context_budget:
                break
            ctx.insert(0, b)
            used += cost
        target = bars[start:end]
        if start == 0:
            target = ["<|piece_start|>", *header, *target]
        if end == len(bars):
            target = [*target, "<|piece_end|>"]
        yield (header if start else None), ctx, "\n".join(target), start + 1
        start = end


def build_examples(text, desc, semitones=(0,), key_dropout=0.3, rng=None, **window_kw):
    """Prompt/completion examples for one piece across transpositions."""
    rng = rng or random.Random(0)
    out = []
    for semis in semitones:
        t = transpose(text, semis) if semis else text
        if t is None:
            continue
        m = meta_fields(t.splitlines()[1])
        for header, ctx, target, next_bar in windows(t, desc.get("sections"), **window_kw):
            variant = rng.choice([v for v in VARIANTS if desc.get(v)])
            key = m.get("key") if rng.random() > key_dropout else None
            meter = m.get("ts") if rng.random() > key_dropout else None
            out.append({
                "prompt": make_prompt(desc[variant], key, meter, header, ctx, next_bar),
                "completion": [{"role": "assistant", "content": target}],
                "file": desc["file"],
                "semis": semis,
            })
    return out


def build_split(texts, descs, files, semitones=(0,), per_piece=None, seed=0, **kw):
    """Examples for `files`, given {file: encoded text} and {file: description record}.
    With `per_piece`, each piece gets the original key plus `per_piece - 1` random shifts."""
    rng = random.Random(seed)
    out = []
    for f in files:
        if f not in texts or f not in descs:
            continue
        shifts = list(semitones)
        if per_piece and per_piece < len(shifts):
            others = [s for s in shifts if s != 0]
            shifts = [0] + rng.sample(others, per_piece - 1)
        out += build_examples(texts[f], descs[f], shifts, rng=rng, **kw)
    rng.shuffle(out)
    return out


def work_id(filename):
    """Group movements of one work: 'appass_1.mid' -> 'appass', 'mz_570_3.mid' -> 'mz_570'."""
    stem = filename.rsplit(".", 1)[0]
    return re.sub(r"_\d+$", "", stem)


def split_by_work(filenames, test_frac=0.1, seed=0):
    works = sorted({work_id(f) for f in filenames})
    random.Random(seed).shuffle(works)
    test_works = set(works[: max(1, round(len(works) * test_frac))])
    train = [f for f in filenames if work_id(f) not in test_works]
    test = [f for f in filenames if work_id(f) in test_works]
    return train, test


In [ ]:
%%writefile midicomposer_lib/midicomposer/compose.py
"""Chunked generation: header + opening bars, then continuations until the piece ends."""

from .dataset import approx_tokens, make_prompt, meta_fields, split_piece


def _complete_bars(text):
    """Bars from model output, dropping a trailing bar that is missing a hand line."""
    _, bars = split_piece(text)
    if bars and len(bars[-1].splitlines()) < 3 and "<|piece_end|>" not in text:
        bars = bars[:-1]
    return bars


def compose(generate_fn, description, key=None, meter=None, max_chunks=40, context_budget=1200,
            max_context_bars=8, measure=approx_tokens, on_chunk=None):
    """`generate_fn(messages) -> str` produces one assistant reply. Returns the full score text."""
    first = generate_fn(make_prompt(description, key, meter))
    header = [l for l in first.splitlines() if l.startswith(("<|meta|>", "<|plan|>"))]
    if not header:
        header = [f"<|meta|> key={key or 'C_Major'} ts={meter or '4/4'} bars=64 spb=12"]
    total = int(meta_fields(header[0]).get("bars", 0) or 0)
    bars = _complete_bars(first)
    done = "<|piece_end|>" in first
    if on_chunk:
        on_chunk(len(bars), total)

    for _ in range(max_chunks - 1):
        if done or (total and len(bars) >= total):
            break
        ctx, used = [], 0
        for b in reversed(bars[-max_context_bars:]):
            if used + measure(b) > context_budget:
                break
            ctx.insert(0, b)
            used += measure(b)
        out = generate_fn(make_prompt(description, key, meter, header, ctx, len(bars) + 1))
        new = _complete_bars(out)
        if not new:
            break
        bars += new
        done = "<|piece_end|>" in out
        if on_chunk:
            on_chunk(len(bars), total)

    return "\n".join(["<|piece_start|>", *header, *bars, "<|piece_end|>"])


In [ ]:
%%writefile midicomposer_lib/midicomposer/evaluate.py
"""Cheap automatic checks for generated scores. Listening is still the real test."""

import io
import re

import numpy as np
import pretty_midi

from .codec import HANDS, _NOTE, decode
from .dataset import meta_fields, split_piece


def to_pretty_midi(text):
    buf = io.BytesIO()
    decode(text).save(file=buf)
    buf.seek(0)
    return pretty_midi.PrettyMIDI(buf)


def pitch_class_hist(pm):
    h = np.zeros(12)
    for inst in pm.instruments:
        for n in inst.notes:
            h[n.pitch % 12] += n.end - n.start
    return h / h.sum() if h.sum() else h


def hand_lines(text):
    return [l for l in text.splitlines() if l.split(" ", 1)[0] in HANDS and len(l) > 2]


def evaluate(text, requested_key=None, reference_text=None, train_lines=None):
    meta, bars = split_piece(text)
    toks = [t for l in hand_lines(text) for t in l.split()[1:]]
    valid = sum(bool(_NOTE.match(t) or re.fullmatch(r"v\d+", t)) for t in toks)
    pm = to_pretty_midi(text)
    out = {
        "bars": len(bars),
        "planned_bars": int(meta_fields(meta).get("bars", 0) or 0),
        "notes": sum(len(i.notes) for i in pm.instruments),
        "valid_token_frac": valid / max(1, len(toks)),
        "duration_s": round(float(pm.get_end_time()), 1),
    }
    if requested_key:
        out["key_match"] = meta_fields(meta).get("key", "").lower() == requested_key.replace(" ", "_").lower()
    if reference_text:
        a, b = pitch_class_hist(pm), pitch_class_hist(to_pretty_midi(reference_text))
        out["pc_hist_l1_to_reference"] = float(np.abs(a - b).sum())
    if train_lines is not None:
        lines = [l for l in hand_lines(text) if len(l) > 40]
        out["copied_line_frac"] = sum(l in train_lines for l in lines) / max(1, len(lines))
    return out


In [ ]:
%%writefile midicomposer_lib/midicomposer/runs.py
"""Generate, save, and log evaluation runs so checkpoints and settings can be compared later.

Layout under `out_dir`:
    results.jsonl                      one line per (file, seed, tag), appended as pieces finish
    <tag>/<piece>__<tag>__s<seed>.mid  generated MIDI (+ .txt score, .wav render if FluidSynth works)
"""

import json
import os
import subprocess
import time

from .codec import decode
from .compose import compose
from .dataset import meta_fields
from .describe import DEFAULT_SOUNDFONT
from .evaluate import evaluate


def render_wav(mid_path, wav_path, soundfont=DEFAULT_SOUNDFONT, sample_rate=22050):
    try:
        subprocess.run(["fluidsynth", "-ni", "-F", wav_path, "-r", str(sample_rate), soundfont, mid_path],
                       check=True, capture_output=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None
    return wav_path if os.path.exists(wav_path) else None


def save_piece(text, out_dir, name, render=True):
    os.makedirs(out_dir, exist_ok=True)
    stem = os.path.join(out_dir, name)
    with open(stem + ".txt", "w") as fh:
        fh.write(text)
    decode(text).save(stem + ".mid")
    wav = render_wav(stem + ".mid", stem + ".wav") if render else None
    return {"txt": stem + ".txt", "mid": stem + ".mid", "wav": wav}


def load_results(path):
    if not os.path.exists(path):
        return []
    with open(path) as fh:
        return [json.loads(line) for line in fh if line.strip()]


def run_eval(generate_fn, files, texts, descs, out_dir, tag, seeds=(0,), variant="styled_detailed",
             max_chunks=40, train_lines=None, set_seed=None, settings=None, render=True, on_piece=None):
    """Compose from each file's brief for each seed, save outputs, and append results.

    Pieces already logged for this tag and seed are skipped, so an interrupted run resumes.
    `set_seed(seed)` is called before each piece so sampling is reproducible.
    """
    os.makedirs(out_dir, exist_ok=True)
    results_path = os.path.join(out_dir, "results.jsonl")
    done = {(r["file"], r["seed"], r["tag"]) for r in load_results(results_path)}
    rows = []
    for f in files:
        if f not in descs or f not in texts:
            continue
        brief = descs[f].get(variant) or descs[f]["brief"]
        meta = meta_fields(texts[f].splitlines()[1])
        key, meter = meta.get("key"), meta.get("ts")
        reference = None
        for seed in seeds:
            if (f, seed, tag) in done:
                continue
            if set_seed:
                set_seed(seed)
            start = time.time()
            text = compose(generate_fn, brief, key=key, meter=meter, max_chunks=max_chunks)
            seconds = round(time.time() - start, 1)
            if reference is None:
                reference = evaluate(texts[f], requested_key=key, train_lines=train_lines)
            row = {
                "tag": tag,
                "file": f,
                "seed": seed,
                "variant": variant,
                "brief": brief,
                "key": key,
                "meter": meter,
                "settings": settings or {},
                "max_chunks": max_chunks,
                "seconds": seconds,
                "metrics": evaluate(text, requested_key=key, reference_text=texts[f], train_lines=train_lines),
                "reference_metrics": reference,
                "paths": save_piece(text, os.path.join(out_dir, tag), f"{f[:-4]}__{tag}__s{seed}", render),
                "created": time.strftime("%Y-%m-%d %H:%M:%S"),
            }
            with open(results_path, "a") as fh:
                fh.write(json.dumps(row) + "\n")
            rows.append(row)
            if on_piece:
                on_piece(row)
    return rows


In [ ]:
import importlib
importlib.invalidate_caches()

In [ ]:
SMOKE_TEST = True

if USE_DRIVE:
    WORK_DIR = '/content/drive/MyDrive/midi-composer'
    MIDI_DIR = '/content/drive/MyDrive/MIDI'
elif IN_COLAB:
    WORK_DIR = '/content/work'
    MIDI_DIR = '/content/MIDI'
else:
    WORK_DIR = os.path.join(REPO_DIR, 'data')
    MIDI_DIR = os.path.expanduser('~/MIDI')
RUN = 'smoke' if SMOKE_TEST else 'full'
TEXT_DIR = f'{WORK_DIR}/text'
AUDIO_DIR = f'{WORK_DIR}/audio'
CKPT_DIR = f'{WORK_DIR}/checkpoints/{RUN}'
ADAPTER_DIR = f'{WORK_DIR}/adapters/{RUN}'
OUT_DIR = f'{WORK_DIR}/outputs/{RUN}'
for d in (TEXT_DIR, AUDIO_DIR, CKPT_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

DESCRIBE_MODEL = 'gemini-2.5-pro'       # any Gemini model that accepts audio
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct'
MAX_LEN = 5120                          # longest training window is ~4.7k tokens
TRANSPOSITIONS = range(-3, 4)           # semitone shifts to sample from
PER_PIECE = 3                           # original key + 2 random shifts per piece
EVAL_SEEDS = [0]                        # one generated piece per test work per seed
EVAL_LIMIT = None                       # None = every held-out work; or an int to cap it

## 1. MIDI files

In [ ]:
if IN_COLAB and not glob.glob(f'{MIDI_DIR}/*.mid'):
    from google.colab import files
    print(f'No MIDI files in {MIDI_DIR}. Upload a .zip of them:')
    os.makedirs(MIDI_DIR, exist_ok=True)
    for name in files.upload():
        !unzip -qoj "{name}" '*.mid' -d "{MIDI_DIR}"

paths = sorted(glob.glob(f'{MIDI_DIR}/*.mid'))
if SMOKE_TEST:
    paths = paths[::max(1, len(paths) // 12)][:12]
names = [os.path.basename(p) for p in paths]
print(len(paths), 'pieces')

## 2. Encode MIDI to text

Format is documented in `src/midicomposer/codec.py`.

In [ ]:
import warnings
import pretty_midi
from midicomposer.codec import encode

warnings.filterwarnings('ignore')
texts = {}
for p, n in zip(paths, names):
    cache = f'{TEXT_DIR}/{n[:-4]}.txt'
    if not os.path.exists(cache):
        with open(cache, 'w') as fh:
            fh.write(encode(pretty_midi.PrettyMIDI(p)))
    texts[n] = open(cache).read()

print(texts[names[0]][:800])

## 3. Composition-brief descriptions

Each piece is rendered to audio and sent to Gemini with its metadata and per-bar analysis. One JSON line per piece is appended to Drive as it goes, so this resumes after a disconnect. The full collection is about 23 hours of audio.

In [ ]:
from midicomposer.describe import describe_all

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if GEMINI_API_KEY:
    from google import genai
    DESC_PATH = f'{WORK_DIR}/descriptions.jsonl'
    describe_all(paths, DESC_PATH, genai.Client(api_key=GEMINI_API_KEY), DESCRIBE_MODEL, AUDIO_DIR)
else:
    print('No GEMINI_API_KEY: using metadata-only descriptions (smoke test only).')
    DESC_PATH = f'{WORK_DIR}/descriptions_fallback.jsonl'
    describe_all(paths, DESC_PATH)

descs = {}
for line in open(DESC_PATH):
    d = json.loads(line)
    descs[d['file']] = d
print(f'{sum(n in descs for n in names)}/{len(names)} described')
print(json.dumps(descs[names[0]], indent=1)[:1500])

## 4. Training windows

Each piece is cut into windows of up to 32 bars. The first window asks for the header (key, meter, length, section plan) and the opening bars; later windows give the header and recent bars and ask for the next ones. Train and test are split by **work**, so every movement of a sonata lands on the same side.

In [ ]:
from midicomposer.dataset import build_split, split_by_work

train_files, test_files = split_by_work(names, test_frac=0.1)
train_examples = build_split(texts, descs, train_files, TRANSPOSITIONS, per_piece=PER_PIECE)
test_examples = build_split(texts, descs, test_files)
json.dump({'train': train_files, 'test': test_files}, open(f'{WORK_DIR}/split_{RUN}.json', 'w'))

approx = sum(len(m['content']) for e in train_examples for m in e['prompt'] + e['completion']) / 1.25
print(f'{len(train_files)} train / {len(test_files)} test pieces')
print(f'{len(train_examples)} training windows, ~{approx / 1e6:.1f}M tokens per epoch')
print(train_examples[0]['prompt'][1]['content'][:700])

## 5. LoRA fine-tune (GPU)

If pip asks you to restart the runtime after installing Unsloth, restart and run all cells again; the cached data makes that quick.

Rough throughput for a 3B model with QLoRA: a T4 manages a few hundred thousand tokens per hour; an A100 is roughly 10x faster. A full run (~20M tokens) is a job for an A100.

In [ ]:
%pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
import torch

print(torch.cuda.get_device_name(0))
model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, random_state=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth')

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([
    {'text': tokenizer.apply_chat_template(e['prompt'] + e['completion'], tokenize=False)}
    for e in train_examples
])

cfg = dict(
    output_dir=CKPT_DIR, dataset_text_field='text', packing=False,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    num_train_epochs=1, max_steps=30 if SMOKE_TEST else -1,
    optim='adamw_8bit', bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
    logging_steps=5, save_steps=200, save_total_limit=2, report_to='none', seed=0,
)
try:
    args = SFTConfig(max_length=MAX_LEN, **cfg)
except TypeError:
    args = SFTConfig(max_seq_length=MAX_LEN, **cfg)
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, args=args)

# Loss only on the score the model writes, not on the brief.
trainer = train_on_responses_only(
    trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')

resume = bool(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
trainer.train(resume_from_checkpoint=True if resume else None)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

## 6. Compose and evaluate

Generation runs window by window until the model writes `<|piece_end|>` or reaches the length it planned. On a T4, a 3B model takes a few minutes per 32-bar window, so the full test set (~30 works) is a job for an A100.

For every held-out work and seed, the eval cell saves to `outputs/<run>/<tag>/`:
- `<piece>__<tag>__s<seed>.mid`, `.txt` (score), and `.wav` (render)

and appends a line to `outputs/<run>/results.jsonl` with the brief, key, meter, sampling settings, metrics, the same metrics for the real piece as a baseline, and the file paths. `<tag>` names the base model, LoRA rank, training step, and temperature, so different checkpoints and settings sit side by side. Pieces already logged for a tag and seed are skipped, so rerunning after a disconnect resumes.

To evaluate a saved adapter in a fresh session, run the setup cells, load it with `FastLanguageModel.from_pretrained(ADAPTER_DIR, max_seq_length=MAX_LEN, load_in_4bit=True)` instead of training, and set `STEP` below to name it (e.g. `'step1800'`).

In [ ]:
from IPython.display import Audio, display
from transformers import set_seed
from midicomposer.evaluate import hand_lines
from midicomposer.runs import run_eval

FastLanguageModel.for_inference(model)
GEN = dict(max_new_tokens=3500, temperature=0.9, top_p=0.95)

def generate_fn(messages):
    ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(input_ids=ids, do_sample=True, **GEN)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

STEP = f'step{trainer.state.global_step}' if 'trainer' in globals() else 'adapter'
TAG = f"{BASE_MODEL.split('/')[-1]}-r32-{STEP}-t{GEN['temperature']}"
train_lines = {l for f in train_files for l in hand_lines(texts[f])}
eval_files = test_files[:1] if SMOKE_TEST else test_files[:EVAL_LIMIT]
max_chunks = 2 if SMOKE_TEST else 40

def show(row):
    m, ref = row['metrics'], row['reference_metrics']
    print(f"{row['file']} seed {row['seed']}: {m['bars']}/{m['planned_bars']} bars, "
          f"valid tokens {m['valid_token_frac']:.3f}, key match {m.get('key_match')}, "
          f"copied lines {m['copied_line_frac']:.3f} (real piece: {ref['copied_line_frac']:.3f}), {row['seconds']}s")
    if row['paths']['wav']:
        display(Audio(row['paths']['wav']))

print(f'Tag {TAG}: {len(eval_files)} works x {len(EVAL_SEEDS)} seeds -> {OUT_DIR}/{TAG}/')
rows = run_eval(generate_fn, eval_files, texts, descs, OUT_DIR, TAG, seeds=EVAL_SEEDS,
                max_chunks=max_chunks, train_lines=train_lines, set_seed=set_seed,
                settings=GEN, on_piece=show)

In [ ]:
import pandas as pd
from midicomposer.runs import load_results

df = pd.json_normalize(load_results(f'{OUT_DIR}/results.jsonl'))
cols = ['metrics.valid_token_frac', 'metrics.bars', 'metrics.planned_bars', 'metrics.key_match',
        'metrics.pc_hist_l1_to_reference', 'metrics.copied_line_frac',
        'reference_metrics.copied_line_frac', 'seconds']
display(df.groupby('tag')[cols].mean(numeric_only=False).round(3))
display(df[['tag', 'file', 'seed'] + cols + ['paths.mid']])

In [ ]:
import time
from midicomposer.compose import compose
from midicomposer.evaluate import evaluate
from midicomposer.runs import save_piece

BRIEF = '''Write a melancholy nocturne that opens with a singing right-hand melody over a
rocking left-hand accompaniment, grows more agitated and chromatic in the middle section,
climaxes around two thirds of the way through, and fades to a quiet, consoling close.'''
KEY, METER, SEED = 'C# minor', '4/4', 0

set_seed(SEED)
text = compose(generate_fn, BRIEF, key=KEY, meter=METER, max_chunks=max_chunks,
               on_chunk=lambda n, total: print(f'  {n}/{total or "?"} bars'))
name = f"custom__{TAG}__s{SEED}__{time.strftime('%Y%m%d-%H%M%S')}"
paths = save_piece(text, f'{OUT_DIR}/custom', name)
metrics = evaluate(text, requested_key=KEY, train_lines=train_lines)
json.dump({'brief': BRIEF, 'key': KEY, 'meter': METER, 'seed': SEED, 'tag': TAG, 'settings': GEN,
           'metrics': metrics, 'paths': paths}, open(f'{OUT_DIR}/custom/{name}.json', 'w'), indent=1)
print(metrics)
if paths['wav']:
    display(Audio(paths['wav']))

In [ ]:
# Without Drive, everything in WORK_DIR disappears when the runtime disconnects: download the outputs.
if IN_COLAB and not USE_DRIVE:
    from google.colab import files
    !cd {WORK_DIR} && zip -qr /content/midi_composer_outputs.zip outputs adapters descriptions*.jsonl
    files.download('/content/midi_composer_outputs.zip')